In [1]:
#Project set up

#All necessary packages
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from torchvision.datasets import CelebA
from torchvision.models import resnet18, ResNet18_Weights
from torchmetrics.classification import BinaryAccuracy, BinaryAUROC, BinaryF1Score
import gc

# Setting seed for reproducability
torch.manual_seed(42)

# Device configuration - will choose CPU for my setup
device = torch.device("cpu")
print(f"Active execution device: {device}")

# Global Hyperparameters
BATCH_SIZE = 64        # Number of training samples passed through the neural network on each forward/backward pass
LEARNING_RATE = 1e-4    # Use smaller learning rate as ResNet is pretrained. Default range for Adam Opt
NUM_EPOCHS = 5          # Number of training passes per part
DATA_DIR = "./data"     

# Target label
TARGET_ATTR = "Smiling"

# 10 Facial Attributes of Interest
CONCEPT_ATTRS = [
    "Mouth_Slightly_Open",
    "High_Cheekbones",
    "Chubby",
    "Narrow_Eyes",
    "Bags_Under_Eyes",
    "Big_Lips",
    "Big_Nose",
    "Pointy_Nose",
    "Bushy_Eyebrows",
    "Arched_Eyebrows",
]

# variable for concept quantity
NUM_CONCEPTS = len(CONCEPT_ATTRS) 

# Crops the image to reduce pixels for compute time and makes it square
transform = transforms.Compose([
    transforms.CenterCrop(178), #Crop first so PyTorch doesn't squish rectangular image into a square with Resize
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
])

# Load our datasets
full_train_ds = CelebA(root=DATA_DIR, split="train", target_type="attr", download=False, transform=transform)
full_test_ds  = CelebA(root=DATA_DIR, split="test",  target_type="attr", download=False, transform=transform)

# Subsampling dataset since I am using CPU
train_dataset = Subset(full_train_ds, list(range(5000)))
test_dataset  = Subset(full_test_ds, list(range(1000)))

# Grabbing the indices for the target and the concepts
target_y_idx = full_train_ds.attr_names.index(TARGET_ATTR)
concept_indices = [full_train_ds.attr_names.index(attr) for attr in CONCEPT_ATTRS]

print(f"Train subset size: {len(train_dataset)} | Test subset size: {len(test_dataset)}")

# Load in a pre-trained ResNet-18 Neural network and remove the final layer.
# Instead, final layer will output a vector of features, which we flatten
# to whatever dimension we need for our problem. Ex. 1 for "Smiling", 10 
# for all features of interest. 
class SharedResNetBackbone(nn.Module):
    """
    Base feature extractor class wrapping an ImageNet-pretrained ResNet-18 backbone.
    Strips off the default 1000-class fc layer, producing a 512-dimensional vector.
    """
    def __init__(self, pretrained=True):
        super(SharedResNetBackbone, self).__init__()
        weights = ResNet18_Weights.DEFAULT if pretrained else None
        resnet = resnet18(weights=weights)
        
        self.in_features = resnet.fc.in_features  # 512 features
        
        # Remove original fc layer by slicing up to AvgPool2d
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])

    def forward(self, x):
        features = self.backbone(x)
        return torch.flatten(features, 1)  # Output shape: (N, 512)

Active execution device: cpu
Train subset size: 5000 | Test subset size: 1000


In [2]:
#Part 1

# Custom collate function to extract 'Smiling' target vector
# This is used because CelebA includes large multidimensional images
# so we define a custom collate_fn to avoid returning all 40 attributes per 
# image and just the specific Smiling target label.
# Returns a tuple of the images and their corresponding target attribute
def baseline_collate_fn(batch):
    images, target_attrs = zip(*batch)
    images = torch.stack(images, dim=0)
    
    # Extract single target 'Smiling' and convert (-1, 1) -> (0.0, 1.0)
    raw_targets = torch.stack([attr[target_y_idx] for attr in target_attrs])
    targets = (raw_targets == 1).float().unsqueeze(1)  # Shape: (N, 1)
    
    return images, targets

# DataLoaders which automate feeding the data into the model.
# Creates a stack of images (size of which is batch_size = 64), where each image is 64x64 pixels with 3 color channels
train_loader_y = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=baseline_collate_fn)
test_loader_y  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=baseline_collate_fn)

# Going to use our pre-defined backbone class so we can 
# re-use the already trained model. Adds on a linear layer 
# so we can use it for our Smiling classification problem
# and produce a single output per image. This is not a probability yet
# because this is done later with BCEWithLogitsLoss() which applies the 
# sigmoid function.
class BaselineClassifier(nn.Module):
    def __init__(self, backbone, num_classes=1):
        super(BaselineClassifier, self).__init__()
        self.backbone = backbone
        self.classifier = nn.Linear(backbone.in_features, num_classes)

    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(features)

# Instantiate Shared Backbone and Model
shared_backbone_1 = SharedResNetBackbone(pretrained=True)
baseline_model = BaselineClassifier(backbone=shared_backbone_1, num_classes=1).to(device)

criterion_y = nn.BCEWithLogitsLoss()  #Combines both the sigmoid function wtih the binary cross-entropy function
optimizer_y = optim.Adam(baseline_model.parameters(), lr=LEARNING_RATE)  #Instead of SGD, Adam computes individual learning rate per parameter

# Training Loop
print("\n--- Training Baseline Classifier (x -> y) ---")
for epoch in range(NUM_EPOCHS):
    #Initialize starting point for this epoch
    baseline_model.train()
    running_loss = 0.0

    #Iterate through the batches of images
    for images, targets in train_loader_y:
        images, targets = images.to(device), targets.to(device)

        #Clearing gradient from the previous batch
        optimizer_y.zero_grad()
        
        logits = baseline_model(images) # Forward pass
        loss = criterion_y(logits, targets) # Loss function
        loss.backward() # Backwards pass
        optimizer_y.step() # Update model weights

        # Tracking total loss for this epoch
        running_loss += loss.item() * images.size(0)

    #Finding average loss for this epoch across the dataset
    epoch_loss = running_loss / len(train_dataset)
    print(f"Epoch [{epoch + 1}/{NUM_EPOCHS}] - Loss: {epoch_loss:.4f}")

# Model Evaluation (Accuracy & AUROC)

# Can use these two functions to track metrics internally rather than doing
# so by hand with a list
test_acc_metric = BinaryAccuracy().to(device) # % of total test images correctly classified as Smiling
test_auroc_metric = BinaryAUROC().to(device)  # Measures how confident the model is in a correct/incorrect answer

# Switch to eval mode so no training behaviors are triggered
baseline_model.eval()

# Iterate through our test images, pass them through the model, apply sigmoid
# function to get a probability of smiling, and assess accuracy across
# entire test dataset
with torch.no_grad(): #Disables gradient calculations since we are evaluating
    for images, targets in test_loader_y:
        images, targets = images.to(device), targets.to(device)
        logits = baseline_model(images)
        probs = torch.sigmoid(logits)
        
        test_acc_metric.update(probs, targets.int())
        test_auroc_metric.update(probs, targets.int())

#Compute and print our total accuracies
print("\n=== Baseline Classifier (x -> y) Report ===")
print(f"Test Accuracy: {test_acc_metric.compute().item() * 100:.2f}%")
print(f"Test AUROC:    {test_auroc_metric.compute().item():.4f}")


--- Training Baseline Classifier (x -> y) ---
Epoch [1/5] - Loss: 0.5238
Epoch [2/5] - Loss: 0.1746
Epoch [3/5] - Loss: 0.0462
Epoch [4/5] - Loss: 0.0164
Epoch [5/5] - Loss: 0.0091

=== Baseline Classifier (x -> y) Report ===
Test Accuracy: 87.00%
Test AUROC:    0.9433


In [3]:
#Part 2

#Similar to previous collate function but instead Extracts the 10 attributes of interest
def concept_collate_fn(batch):
    images, target_attrs = zip(*batch)
    images = torch.stack(images, dim=0)
    
    # Extract all 10 concept values
    raw_concepts = torch.stack([
        torch.tensor([attr[idx] for idx in concept_indices]) 
        for attr in target_attrs
    ])
    concepts = (raw_concepts == 1).float()
    
    return images, concepts

# Instantiate DataLoaders with new collate function
train_loader_c = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=concept_collate_fn)
test_loader_c  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=concept_collate_fn)

# Similar class as previous but now the linear head has 10 attributes instead of 1
class ConceptPredictor(nn.Module):
    def __init__(self, backbone, num_concepts=10):
        super(ConceptPredictor, self).__init__()
        self.backbone = backbone
        self.concept_head = nn.Linear(backbone.in_features, num_concepts)

    def forward(self, x):
        features = self.backbone(x)
        return self.concept_head(features)  # Shape: (N, 10)

# Initialize the model with the shared backbone and 10 attributes of interest
shared_backbone_2 = SharedResNetBackbone(pretrained=True)
concept_model = ConceptPredictor(backbone=shared_backbone_2, num_concepts=NUM_CONCEPTS).to(device)

# Defining the criterion and omptimizer as discussed before 
criterion_c = nn.BCEWithLogitsLoss() # Able to handle multi-label targets across all 10 outputs
optimizer_c = optim.Adam(concept_model.parameters(), lr=LEARNING_RATE)

print("\n--- Training Concept Predictor (x -> c) ---")

#Same steps for the epochs as before, except calculating loss across all 10 attributes
for epoch in range(NUM_EPOCHS):
    concept_model.train()
    running_loss = 0.0
    
    for images, concept_targets in train_loader_c:
        images, concept_targets = images.to(device), concept_targets.to(device)
        
        optimizer_c.zero_grad()
        logits = concept_model(images)
        loss = criterion_c(logits, concept_targets)
        loss.backward()
        optimizer_c.step()
        
        running_loss += loss.item() * images.size(0)
        
    epoch_loss = running_loss / len(train_dataset)
    print(f"Epoch [{epoch + 1}/{NUM_EPOCHS}] - Loss: {epoch_loss:.4f}")


# Setup metrics as before, but will use lists to evaluate each concept individually
acc_metric = BinaryAccuracy().to(device)
f1_metric = BinaryF1Score().to(device)

# Lists to collect predictions and targets per-attribute
all_preds, all_targets = [], []

# Same eval set up as before
concept_model.eval()
with torch.no_grad():
    for images, concept_targets in test_loader_c:
        images, concept_targets = images.to(device), concept_targets.to(device)
        logits = concept_model(images)
        probs = torch.sigmoid(logits)

        #Append the 10 attribute performances to the lists
        all_preds.append(probs)
        all_targets.append(concept_targets.int())

# Concatenate all the batches together so that we have performance across attributes for all batches in matrix form
all_preds = torch.cat(all_preds, dim=0)
all_targets = torch.cat(all_targets, dim=0)

# Now to iterate through each attribute of interest, look at the predict performances, and calculate
# the metrics so we get performance across every attribute for all of the data.
per_concept_accs, per_concept_f1s = [], []

print("\n--- Per-Concept Performance ---")
for i, concept_name in enumerate(CONCEPT_ATTRS):
    c_preds, c_targets = all_preds[:, i], all_targets[:, i]
    
    c_acc = acc_metric(c_preds, c_targets).item()
    c_f1 = f1_metric(c_preds, c_targets).item()
    
    per_concept_accs.append(c_acc)
    per_concept_f1s.append(c_f1)
    
    print(f"{concept_name:<22} | Acc: {c_acc * 100:.2f}% | F1: {c_f1:.4f}")

# Compute macro-average across all 10 attributes
mean_acc = sum(per_concept_accs) / NUM_CONCEPTS
mean_f1  = sum(per_concept_f1s) / NUM_CONCEPTS

print("\n=== Concept Predictor (x -> c) Report ===")
print(f"Mean Concept Accuracy (Macro): {mean_acc * 100:.2f}%")
print(f"Mean Concept F1 Score (Macro): {mean_f1:.4f}")


--- Training Concept Predictor (x -> c) ---
Epoch [1/5] - Loss: 0.5188
Epoch [2/5] - Loss: 0.3477
Epoch [3/5] - Loss: 0.2572
Epoch [4/5] - Loss: 0.1781
Epoch [5/5] - Loss: 0.1227

--- Per-Concept Performance ---
Mouth_Slightly_Open    | Acc: 78.90% | F1: 0.7849
High_Cheekbones        | Acc: 77.40% | F1: 0.7656
Chubby                 | Acc: 93.50% | F1: 0.2169
Narrow_Eyes            | Acc: 84.50% | F1: 0.1040
Bags_Under_Eyes        | Acc: 79.20% | F1: 0.3459
Big_Lips               | Acc: 63.60% | F1: 0.2288
Big_Nose               | Acc: 80.70% | F1: 0.4683
Pointy_Nose            | Acc: 69.70% | F1: 0.3674
Bushy_Eyebrows         | Acc: 87.00% | F1: 0.3500
Arched_Eyebrows        | Acc: 76.00% | F1: 0.5312

=== Concept Predictor (x -> c) Report ===
Mean Concept Accuracy (Macro): 79.05%
Mean Concept F1 Score (Macro): 0.4163


Interpretation of above results: While accuracy seems high across all attributes, the F1 score tells us that the model isn't actually doing a good job classifying, it is just getting lucky that alot of the data in the dataset just doesnt contain images with those attributes present. For example, if it classifies an image as "not chubby", it tends to be correct because only a small subset of the data actually contains images labeled "chubby". The F1 scores basically confirms that the model was well trained on "Mouth Slightly Open" and "High Cheekbones" likely because there are ample examples of these in our training dataset, so the model got good at accurately detecting it. The other attributes however, the model does a poor job of detecting them, but gets lucky due to the proportionaly disparity in the test set of what images actually have what features.

## Part 3: Concept Bottleneck Model
Goal: So far the internals of the model are a black box in terms of how it is deciding whether a person is smiling or not. For this part, we are going to separate the classification problem into two steps. The first being allow the neural network to provide probabilities of the 10 attributes of interest as in part 2. Then, use another layer that takes those ten attributes in as an input, and creates a prediction on whether the person is smiling. Basically forcing some transparency into the model by seeing how certain features become indicative of someone smiling.

In [4]:
# Same as aforementioned classes but this takes the previously defined and
# trained model, freezes the parameters of that model so it does not update,
# and puts on a linear layer that will be trained on the output of c to predict y
class IndependentCBM(nn.Module):
    def __init__(self, concept_predictor, num_concepts=10):
        super(IndependentCBM, self).__init__()
        # Feature 1: Pre-trained x -> c concept predictor
        self.concept_predictor = concept_predictor
        
        # Freeze x -> c so parameter do not update
        for param in self.concept_predictor.parameters():
            param.requires_grad = False
            
        # Feature 2: Bottleneck layer mapping predicted concepts to target (c -> y)
        self.c_to_y = nn.Linear(num_concepts, 1)

    def forward(self, x):
        # 1. Get concept logits from frozen predictor
        concept_logits = self.concept_predictor(x)
        
        # 2. Convert logits to probabilities (0.0 to 1.0)
        concept_probs = torch.sigmoid(concept_logits)
        
        # 3. Predict target label (y) strictly from concept probabilities
        target_logit = self.c_to_y(concept_probs)
        return target_logit

# Instantiate Independent CBM using the trained concept_model from Part 2
cbm_model = IndependentCBM(concept_predictor=concept_model, num_concepts=10).to(device)


# Only using optomizer to train the parameters of the c_to_y linear layer
optimizer_cbm = torch.optim.Adam(cbm_model.c_to_y.parameters(), lr=1e-3)
criterion_cbm = nn.BCEWithLogitsLoss()


# We reuse the original baseline collate because target is 'Smiling' (y)
train_loader_cbm = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=baseline_collate_fn
)
test_loader_cbm = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=baseline_collate_fn
)


# Iterating through 5 epochs of training the model
# For all of the images in the training dataset, we are going to pass 
# the images through the pretrained model, which will output 10 attributes
# with probabilities per attribute. These will then get passed into the final
# layer to predict whether the person is smiling. Loss function will be
# performed on these final predictions to train the model.
print("\n--- Training Independent CBM (c -> y Head) ---")
for epoch in range(NUM_EPOCHS):
    #Trains the second step but freezes the first part into eval mode
    cbm_model.train()
    cbm_model.concept_predictor.eval()

    #Keep track of the loss of each epoch
    running_loss = 0.0
    
    for images, targets in train_loader_cbm:
        images, targets = images.to(device), targets.to(device)
        
        optimizer_cbm.zero_grad()
        #Grabbing the ouput of the first part of the model
        logits = cbm_model(images)
        #Calculating the loss of the second part, backward propogating, and running the optimizer
        loss = criterion_cbm(logits, targets)
        loss.backward()
        optimizer_cbm.step()
        
        running_loss += loss.item() * images.size(0)
        
    epoch_loss = running_loss / len(train_dataset)
    print(f"Epoch [{epoch + 1}/{NUM_EPOCHS}] - CBM Loss: {epoch_loss:.4f}")


# Now to calculate the evaluation metrics of how the model is performing
cbm_acc_metric = BinaryAccuracy().to(device)
cbm_auroc_metric = BinaryAUROC().to(device)

#Same technique as discussed before, testing model output to true trest results
cbm_model.eval()
with torch.no_grad():
    for images, targets in test_loader_cbm:
        images, targets = images.to(device), targets.to(device)

        #Grab the logits, create the probability, and check if the prediction is correct
        logits = cbm_model(images)
        probs = torch.sigmoid(logits)
        
        cbm_acc_metric.update(probs, targets.int())
        cbm_auroc_metric.update(probs, targets.int())

cbm_acc = cbm_acc_metric.compute().item() * 100
cbm_auroc = cbm_auroc_metric.compute().item()


--- Training Independent CBM (c -> y Head) ---
Epoch [1/5] - CBM Loss: 0.7418
Epoch [2/5] - CBM Loss: 0.7158
Epoch [3/5] - CBM Loss: 0.6950
Epoch [4/5] - CBM Loss: 0.6769
Epoch [5/5] - CBM Loss: 0.6597


In [5]:
print("\n==================================================")
print("=== Independent CBM (x -> c -> y) Final Report ===")
print("==================================================")
print(f"CBM Test Accuracy: {cbm_acc:.2f}%")
print(f"CBM Test AUROC:    {cbm_auroc:.4f}")
print("--------------------------------------------------")

# Accessing baseline values computed in Part 1 for direct comparison
print("=== Performance Comparison ===")
print(f"Baseline (x -> y) Accuracy : {test_acc_metric.compute().item() * 100:.2f}%  |  CBM Accuracy : {cbm_acc:.2f}%")
print(f"Baseline (x -> y) AUROC    : {test_auroc_metric.compute().item():.4f}  |  CBM AUROC    : {cbm_auroc:.4f}")
print("==================================================")


=== Independent CBM (x -> c -> y) Final Report ===
CBM Test Accuracy: 69.60%
CBM Test AUROC:    0.7598
--------------------------------------------------
=== Performance Comparison ===
Baseline (x -> y) Accuracy : 87.00%  |  CBM Accuracy : 69.60%
Baseline (x -> y) AUROC    : 0.9433  |  CBM AUROC    : 0.7598


#### Part 3 Interpretation of Results
Because I have heavily subsetted the data to only 5000 images in training, we saw in the previous part that the model was not doing a good job predicting attributes that were less commonly found in the training data. So the saying of garbage in and garbage out is true in this case because we are training a final layer on prediciting whether someone is smiling based on attributes that the model isn't confidently or accurately outputting. To address this, we need to read in more data

However, there is also the fact that bottleneck models structured like this will always suffer a slight degradation in results from exclusively predicting whether someone is smiling from all the data in the image because the bottleneck model is constrained to only the 10 input attributes and thus can miss other insights within the image that could be indicative of someone smiling.

## Part 4: Adding Side Channel
To avoid the loss of data that is involved within a bottleneck model, we are going to add a sidechannel so that the second layer can be trained on both things: the 10 attributes as an input, and then the entire image data as well. These two results will then be summed into the final logit result.

To create the side channel, we need to use a "Hook" that allows us to grab a tensor within the neural network before it gets passed through the layer that outputs 10 logits for the attributes of interest. Once we grab this, we can place it through our side channel so that we can also predict whether or not the person is smiling based on ALL the information of the image after the neural network has interpreted it.

In [6]:
class HybridCBM(nn.Module):
    def __init__(self, concept_predictor, num_concepts=10, feature_dim=512):
        super(HybridCBM, self).__init__()
        self.concept_predictor = concept_predictor
        
        # Freeze the concept predictor parameters again
        for param in self.concept_predictor.parameters():
            param.requires_grad = False
            
        # Pathway 1: f(c) -> Concept-based linear classifier (10 -> 1)
        self.f_c = nn.Linear(num_concepts, 1)
        
        # Pathway 2: s(x) -> Side-channel linear classifier on backbone features (512 -> 1)
        self.s_x = nn.Linear(feature_dim, 1)
        
        # Buffer to store intermediate 512-dim features captured by the hook
        self.extracted_features = None
        
        def hook_fn(module, input, output):
            # Intercept tensor right after pooling and flatten to (batch_size, 512)
            self.extracted_features = torch.flatten(output, 1)

        # Dynamic Hook Attachment: Traverses concept_predictor to find avgpool automatically
        # Basically finds the layer right before classification occurrs, where the image has been processed fully 
        # by the neural network and we still have all of the information from the image.
        hook_registered = False
        for name, module in self.concept_predictor.named_modules():
            if isinstance(module, nn.AdaptiveAvgPool2d) or 'avgpool' in name.lower():
                module.register_forward_hook(hook_fn)
                hook_registered = True
                print(f"[HybridCBM] Successfully hooked into backbone layer: '{name}'")
                break
                
        if not hook_registered:
            raise RuntimeError("Could not automatically locate 'avgpool' in concept_predictor submodules.")

    def forward(self, x):
        # 1. Run concept predictor forward pass (populates self.extracted_features via hook)
        concept_logits = self.concept_predictor(x)
        concept_probs = torch.sigmoid(concept_logits)
        
        # 2. Extract captured 512-dim features
        features = self.extracted_features
        
        # 3. Calculate logits for both pathways
        f_c_logit = self.f_c(concept_probs)  # Shape: (N, 1)
        s_x_logit = self.s_x(features)       # Shape: (N, 1)
        
        # 4. Sum logits to get final decision
        return f_c_logit + s_x_logit
# Instantiate Hybrid CBM using frozen concept_model from Part 2
hybrid_cbm = HybridCBM(concept_predictor=concept_model, num_concepts=10, feature_dim=512).to(device)


# Only optimize the parameters of the two linear heads: f(c) and s(x)
trainable_params = list(hybrid_cbm.f_c.parameters()) + list(hybrid_cbm.s_x.parameters())
optimizer_hybrid = torch.optim.Adam(trainable_params, lr=1e-3)
criterion_hybrid = nn.BCEWithLogitsLoss()

# DataLoaders reuse target task loaders (Smiling)
train_loader_hybrid = train_loader_cbm
test_loader_hybrid = test_loader_cbm

print("\n--- Training Hybrid CBM (Concept Head + Side Channel) ---")
for epoch in range(NUM_EPOCHS):
    hybrid_cbm.train()
    hybrid_cbm.concept_predictor.eval()  # Keep BatchNorm fixed in backbone
    
    running_loss = 0.0
    for images, targets in train_loader_hybrid:
        images = images.to(device)
        targets = targets.to(device).float().unsqueeze(1) if targets.ndim == 1 else targets.to(device).float()
        
        optimizer_hybrid.zero_grad()
        logits = hybrid_cbm(images)
        loss = criterion_hybrid(logits, targets)
        loss.backward()
        optimizer_hybrid.step()
        
        running_loss += loss.item() * images.size(0)
        
    epoch_loss = running_loss / len(train_loader_hybrid.dataset)
    print(f"Epoch [{epoch + 1}/{NUM_EPOCHS}] - Hybrid CBM Loss: {epoch_loss:.4f}")


hybrid_acc_metric = BinaryAccuracy().to(device)
hybrid_auroc_metric = BinaryAUROC().to(device)

hybrid_cbm.eval()
with torch.no_grad():
    for images, targets in test_loader_hybrid:
        images = images.to(device)
        targets = targets.to(device).int()
        
        logits = hybrid_cbm(images)
        probs = torch.sigmoid(logits)
        
        hybrid_acc_metric.update(probs, targets)
        hybrid_auroc_metric.update(probs, targets)

hybrid_acc = hybrid_acc_metric.compute().item() * 100
hybrid_auroc = hybrid_auroc_metric.compute().item()

[HybridCBM] Successfully hooked into backbone layer: 'backbone.backbone.8'

--- Training Hybrid CBM (Concept Head + Side Channel) ---
Epoch [1/5] - Hybrid CBM Loss: 0.4200
Epoch [2/5] - Hybrid CBM Loss: 0.3537
Epoch [3/5] - Hybrid CBM Loss: 0.3442
Epoch [4/5] - Hybrid CBM Loss: 0.3379
Epoch [5/5] - Hybrid CBM Loss: 0.3288


In [11]:
print("\n==================================================================")
print("=== Hybrid CBM (with Side Channel) Final Report ===")
print("==================================================================")
print(f"Hybrid CBM Test Accuracy : {hybrid_acc:.2f}%")
print(f"Hybrid CBM Test AUROC    : {hybrid_auroc:.4f}")
print("------------------------------------------------------------------")
print("=== Complete Model Performance Comparison ===")
print(f"1. Baseline (x -> y)         : Acc = {test_acc_metric.compute().item() * 100:.2f}% | AUROC = {test_auroc_metric.compute().item():.4f}")
print(f"2. Independent CBM (x->c->y) : Acc = {cbm_acc:.2f}% | AUROC = {cbm_auroc:.4f}")
print(f"3. Hybrid CBM (f(c) + s(x))  : Acc = {hybrid_acc:.2f}% | AUROC = {hybrid_auroc:.4f}")
print("==================================================================")


=== Hybrid CBM (with Side Channel) Final Report ===
Hybrid CBM Test Accuracy : 81.90%
Hybrid CBM Test AUROC    : 0.9033
------------------------------------------------------------------
=== Complete Model Performance Comparison ===
1. Baseline (x -> y)         : Acc = 87.00% | AUROC = 0.9433
2. Independent CBM (x->c->y) : Acc = 69.60% | AUROC = 0.7598
3. Hybrid CBM (f(c) + s(x))  : Acc = 81.90% | AUROC = 0.9033
